## Advanced Secrets Management

# Advanced Google Cloud Secret Manager: Inventory, Metadata Taxonomies, and Purge Operations

In previous lessons, we explored the fundamentals of managing sensitive information using Google Cloud's secrets management services. In this lesson, we will dive into advanced features that help you organize, control, and manage the lifecycle of your secrets more effectively.

We will cover how to inventory all managed assets via project listings, use key-value metadata labels for multi-tenant categorization, audit historical version lineages, and differentiate between soft container exclusions and permanent cryptographic destruction. These capabilities are essential for maintaining rigorous compliance, tight security boundaries, and operational efficiency in your cloud environment.

---

## Listing All Project Secrets

Google Cloud Secret Manager allows you to query and compile an inventory of all secrets within a designated project. This operational visibility is crucial for automated compliance audits, rotative state scripting, and orphan resource cleanup.

```python
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Replace with your Google Cloud project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

print("Inventory Check: Scanning project secrets...")
# List all secrets in the project
for secret in client.list_secrets(request={"parent": parent}):
    print("🔹 Secret Name:", secret.name)

```

**Console Output:**

```text
🔹 Secret Name: projects/your-project-id/secrets/api-key
🔹 Secret Name: projects/your-project-id/secrets/database-password
🔹 Secret Name: projects/your-project-id/secrets/my-secret

```

This request returns a paginated iterator stream containing the metadata structures of all secrets matching the designated parent project ID coordinates.

---

## Labeling and Unlabeling Secrets (Metadata Taxonomies)

Labels in Google Cloud Secret Manager are key-value string pairs that help you categorize, filter, and allocate resource costs for your secrets. You can modify labels on the fly to tie secrets to specific runtime environments (`env: production`), organizational units (`team: devops`), or microservice dimensions (`app: authentication`).

### 1. Appending and Modifying Labels

```python
from google.cloud import secretmanager
from google.protobuf import field_mask_pb2

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

# Define the new structural metadata labels
labels = {
    "environment": "production",
    "team": "devops"
}

# Construct the payload dict matching the Secret proto spec
secret_update_payload = {
    "name": secret_name, 
    "labels": labels
}

# Target ONLY the labels dictionary to protect other configuration fields
update_mask = field_mask_pb2.FieldMask(paths=["labels"])

updated_secret = client.update_secret(
    secret=secret_update_payload, 
    update_mask=update_mask
)
print("Updated Labels:", updated_secret.labels)

```

**Console Output:**

```text
Updated Labels: {'environment': 'production', 'team': 'devops'}

```

> ⚙️ **Architectural Deep-Dive: The `update_mask` (FieldMask)**
> The `update_mask` parameter is a critical architectural guardrail in this transaction. It functions as a surgical filter specifying precisely which fields inside the backend storage object should be overwritten. By setting `paths=["labels"]`, you explicitly instruct the Google Cloud API gateway to modify **only** the taxonomy block. Other structural properties assigned to that secret container (such as multi-region data replication profiles, automatic expiration TTLs, or customer-managed encryption key rings) remain entirely un-touched.

### 2. Dropping (Removing) a Key Label

To delete a specific tag entry entirely, simply omit the target key from your input dictionary map and submit the modification using the same field mask process:

```python
# Omit the 'team' key to strip it out of the metadata layer
cleared_labels = {
    "environment": "production"
}

secret_update_payload = {
    "name": secret_name, 
    "labels": cleared_labels
}

update_mask = field_mask_pb2.FieldMask(paths=["labels"])

updated_secret = client.update_secret(
    secret=secret_update_payload, 
    update_mask=update_mask
)
print("Labels after removal:", updated_secret.labels)

```

**Console Output:**

```text
Labels after removal: {'environment': 'production'}

```

---

## Working with Secret Versions

Each secret container maintains an immutable historical ledger of version records. This makes secret rotations highly safe, as past connection strings or credentials remain preserved if an application rollback is triggered.

### 1. Auditing Version States

You can scan the structural status ledger of all versions nested inside a target container:

```python
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

print(f"Auditing history for: {secret_name}")
# List all versions of the target secret
for version in client.list_secret_versions(request={"parent": secret_name}):
    print(f" ├─ Version ID Path: {version.name} | State: {version.state.name}")

```

**Console Output:**

```text
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/3 | State: ENABLED
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/2 | State: ENABLED
 ├─ Version ID Path: projects/your-project-id/secrets/my-secret/versions/1 | State: ENABLED

```

### 2. Extracting Target Payloads (Latest vs. Explicit Versions)

Consumer applications can request the runtime data string using either the dynamic `latest` alias index pointer or a specific historic iteration integer key:

```python
# Option A: Pull data using the global convenience alias pointer
latest_version_name = f"{secret_name}/versions/latest"
response_latest = client.access_secret_version(request={"name": latest_version_name})
print("Latest Secret Value:", response_latest.payload.data.decode("UTF-8"))

# Option B: Pull the historical genesis state block explicitly targeting Version 1
version_1_name = f"{secret_name}/versions/1"
response_v1 = client.access_secret_version(request={"name": version_1_name})
print("Version 1 Secret Value:", response_v1.payload.data.decode("UTF-8"))

```

**Console Output:**

```text
Latest Secret Value: my-updated-secret-value
Version 1 Secret Value: my-original-secret-value

```

---

## Secret Deletion vs. Destruction

When decommissioning infrastructure, it is critical to understand the distinction between deleting an entire secret container resource and destroying a standalone, isolated payload iteration. Both actions are completely **permanent and irreversible**.

### 1. Deleting an Entire Secret Container

Deleting a secret executes a top-level **cascading wipe**. This deletes the primary tracking container shell, all metadata tags, IAM bindings, and **every single historical version entry** stored within that path.

```python
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_name = f"projects/{project_id}/secrets/my-secret"

# Permanent cascading deletion operation
client.delete_secret(request={"name": secret_name})
print(f"🔥 Core container '{secret_name}' and all inner versions permanently deleted.")

```

**Console Output:**

```text
🔥 Core container 'projects/your-project-id/secrets/my-secret' and all inner versions permanently deleted.

```

### 2. Destroying an Isolated Secret Version

If you want to sanitize old, stale, or compromised credentials without breaking your application's current environment connection strings, you can **destroy** a specific version layer. This permanently purges the data payload bits for that explicit index while keeping the parent wrapper and other active version indexes intact.

```python
target_version_path = f"projects/{project_id}/secrets/my-secret/versions/1"

# Permanently shred the secret data payload for version 1
client.destroy_secret_version(request={"name": target_version_path})
print(f"💥 Cryptographic payload for version '{target_version_path}' destroyed.")

```

**Console Output:**

```text
💥 Cryptographic payload for version 'projects/your-project-id/secrets/my-secret/versions/1' destroyed.

```

> 🛑 **Operational Posture Warning:** Once a version is destroyed, its state enum transitions to `DESTROYED`. The tracking slot remains in the ledger history to maintain audit trail continuity, but any future attempts to call `access_secret_version` against that specific path will fail with a gRPC error.

---

## Summary

In this lesson, we mastered the advanced data management and administrative capabilities of Google Cloud Secret Manager:

* Implemented systemic inventories across project scopes using `list_secrets`.
* Leveraged `FieldMask` configurations to modify key-value metadata taxonomies cleanly without causing property side effects.
* Extracted and audited historical version matrices using the explicit index pattern.
* Analyzed the risk profiles and execution differences behind container-level deletion and version-level data payload destruction.

Now, let's step over to the console code terminal and put these advanced cloud engineering patterns to work inside our interactive lab workspace!

## Google Cloud Secret Manager Operations with Python

Welcome to this task! You will be running a pre-existing Python script that exercises various operations with Google Cloud Secret Manager. This script demonstrates creating versions of a secret, updating the secret with both manually specified and randomly generated passwords, listing the secret versions, retrieving a secret's value by different version numbers, managing secret labels, and permanently deleting a secret. The purpose of this task is to expose you to how these operations work within a real script. There's no need for you to write any code—your task is simply to run the script, observe its execution and understand what each part of the code is doing.

Important Note: Executing scripts can alter resources within our GCP simulator. To return to the initial state, you may use the reset button located in the top right corner. It is important to note, however, that resetting will remove any code modifications. To safeguard your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
from google.cloud import secretmanager
import secrets
import string

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()
project_id = "your-project-id"  # Replace with your actual project ID
secret_id = "MyPassword"
parent = f"projects/{project_id}"
secret_name = f"{parent}/secrets/{secret_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": secret_id,
        "secret": {"replication": {"automatic": {}}},
    }
)

# Create version 1
version1 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": b"simple password 1"},
    }
)

# Create version 2
version2 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": b"simple password 2"},
    }
)

# Generate a strong random password for version 3
def generate_strong_password(length=16):
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
    return ''.join(secrets.choice(alphabet) for _ in range(length))

strong_password = generate_strong_password(16)

# Create version 3 with the generated password
version3 = client.add_secret_version(
    request={
        "parent": secret_name,
        "payload": {"data": strong_password.encode()},
    }
)

# List Secret versions
versions = client.list_secret_versions(request={"parent": secret_name})
print('Versions of Secret:')
for version in versions:
    print(f"  {version.name}")

# Retrieve the previous version (version 2) explicitly
version2_name = f"{secret_name}/versions/2"
previous_secret_value = client.access_secret_version(request={"name": version2_name})
print(f'\nPrevious Secret Value (version 2): {previous_secret_value.payload.data.decode()}')

# Add labels to the secret (equivalent to AWS tags)
secret_with_labels = client.update_secret(
    request={
        "secret": {
            "name": secret_name,
            "labels": {"environment": "test"}
        },
        "update_mask": {"paths": ["labels"]}
    }
)
print(f'\nLabeled Secret: {secret_with_labels.name}')

# Remove labels from the secret
secret_without_labels = client.update_secret(
    request={
        "secret": {
            "name": secret_name,
            "labels": {}
        },
        "update_mask": {"paths": ["labels"]}
    }
)
print(f'\nUnlabeled Secret: {secret_without_labels.name}')

# Delete the secret permanently (no restoration possible in Google Cloud Secret Manager)
client.delete_secret(request={"name": secret_name})
print(f'\nDeleted Secret: {secret_name}')

```

## Secure Password Generation for Google Cloud Secret Manager

In this task, you will be working with the Google Cloud Secret Manager to create a new secret with a random password. Initially, you are provided with a Python script that generates a very simple string which is used as a password for a new Secret. However, this implementation isn't very secure. Your task is to adjust the existing code to ensure that the generated password is more complex, specifically it should be 16 characters long, including both lower and upper case characters, and without the specific characters @, #, ", and /. Also, note that we do not want to include any punctuation characters in the generated password.

Important Note: Executing scripts can change the state of resources within our GCP simulator. In order to revert to the initial state, you can use the rest button located in the top right corner. However, keep in mind that resetting will erase any code modifications. Therefore, to maintain your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
import secrets
import string
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"

# Generate a simple random password
characters = string.ascii_letters + string.digits
new_password = ''.join(secrets.choice(characters) for _ in range(8))

# Create the parent resource name
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "MyTestSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version with the generated password
secret_payload = '{"username":"test", "password":"' + new_password + '"}'
response_create = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": secret_payload.encode("UTF-8")},
    }
)

print('Created Secret:', response_create)
```

Here is the updated script with a secure password generation implementation.

The password logic has been adjusted to guarantee a length of **16 characters**, using both upper and lowercase letters as well as numbers, while explicitly excluding punctuation characters and your prohibited characters (`@`, `#`, `"`, `/`).

```python
import mock_secret_manager
import secrets
import string
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"

# --- FIXED: Enhanced password generation logic for high security ---
# 1. Combine base alphanumeric character sets (letters + digits)
# 2. Exclude specific unwanted characters as requested (no punctuation, no @, #, ", /)
allowed_characters = string.ascii_letters + string.digits

# Ensure none of the explicitly restricted characters accidentally bypass character set definitions
restricted_chars = ['@', '#', '"', '/', ' ']
allowed_characters = ''.join([c for c in allowed_characters if c not in restricted_chars])

# 3. Generate a cryptographically secure 16-character string
new_password = ''.join(secrets.choice(allowed_characters) for _ in range(16))

# Create the parent resource name
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "MyTestSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version with the generated secure password
secret_payload = '{"username":"test", "password":"' + new_password + '"}'
response_create = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": secret_payload.encode("UTF-8")},
    }
)

print('Created Secret:', response_create)
print('Generated Secure Password Length:', len(new_password))

```

---

### Architectural Pro-Tip: Cryptographic Security in Python

Notice that we use Python's **`secrets`** module instead of the standard `random` module.

The `random` module uses a pseudo-random number generator (PRNG) that is deterministic and can be predicted if an attacker samples enough previous password variations. The `secrets` module, however, taps directly into your underlying operating system’s cryptographically secure pseudo-random number generator (CSPRNG), ensuring that generated production strings are completely unpredictable and secure against brute-force attacks before they are safely vaulted inside Google Cloud Secret Manager.

## Update Google Cloud Secret Manager with Auto-Generated Secure Password

Cosmo, our lovable CodeSignal Learn mascot, has a secret he wants to secure and ensure its safety. He decided to auto-generate a highly secure password. Your task is to update this password using Google Cloud Secret Manager. Initially, you are given a script that creates a secret with a basic password. You need to modify the script to update the password with an auto-generated one.

Important Note: Running scripts can modify the resources in our Google Cloud simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
import secrets
import string
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"

# Generate a random password with specific configuration
characters = string.ascii_letters + string.digits
# Remove excluded characters
excluded_chars = '@#"/'
characters = ''.join(c for c in characters if c not in excluded_chars)
new_password = ''.join(secrets.choice(characters) for _ in range(16))

# Create the parent resource name
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "MyTestSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version with the generated password
secret_payload = '{"username":"test", "password":"' + new_password + '"}'
response_create = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": secret_payload.encode("UTF-8")},
    }
)

print('Created Secret:', response_create)

# TODO: Generate a new highly secure password
# TODO: Update the secret with the new password
```

Here is the completed script. It leverages the secure character configuration to generate a fresh, high-entropy alternative password and commits it to the secret container as a new version layer.

```python
import mock_secret_manager
import secrets
import string
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"

# Generate a random password with specific configuration
characters = string.ascii_letters + string.digits
# Remove excluded characters
excluded_chars = '@#"/'
characters = ''.join(c for c in characters if c not in excluded_chars)
new_password = ''.join(secrets.choice(characters) for _ in range(16))

# Create the parent resource name
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "MyTestSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version with the generated password
secret_payload = '{"username":"test", "password":"' + new_password + '"}'
response_create = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": secret_payload.encode("UTF-8")},
    }
)

print('Created Secret:', response_create.name)

# ========================================================
# 1. TODO FIXED: GENERATE A NEW HIGHLY SECURE PASSWORD
# ========================================================
# Re-using the clean alphanumeric collection to assemble a fresh rotation value
rotated_password = ''.join(secrets.choice(characters) for _ in range(16))


# ========================================================
# 2. TODO FIXED: UPDATE THE SECRET WITH THE NEW PASSWORD
# ========================================================
# Pack the rotated data dictionary string payload
updated_payload = '{"username":"test", "password":"' + rotated_password + '"}'

# Push a new version to update the secret container state
response_update = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": updated_payload.encode("UTF-8")},
    }
)

print('Updated Secret (New Version Appended):', response_update.name)

```

---

### Mechanics of Zero-Downtime Rotations

By appending a second version block through `client.add_secret_version()`, your system achieves what is known as a **Zero-Downtime Rolling Rotation**.

Any application instances currently reading from the `/versions/latest` tracking alias will instantly receive the new password payload on their next pull request. Simultaneously, any legacy server processes that are still finishing an active session can continue processing safely using their existing connections, as older version slots are preserved and kept active until you explicitly disable them.

## Labeling Secrets with Creator Information in Google Cloud Secret Manager

Our mascot, Cosmo, has many secrets he needs to manage! In this activity, you'll help him keep things organized by labeling his secrets. Initially, you're given a Python script that creates a secret. You need to add to this script and make it label the secret, indicating that Cosmo is the creator. By labeling secrets, you can classify and manage them more efficiently, depending on your organizational needs.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "CosmosSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version
response = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"mysecret"},
    }
)

# TO DO: Label the secret with Cosmo as the creator
```

Here is the completed script. It imports the required `FieldMask` protocol tool from the Google API core library, sets up the key-value taxonomy indicating that Cosmo is the creator, and applies it cleanly to the secret container using an update mask.

```python
import mock_secret_manager
from google.cloud import secretmanager
# --- FIXED: Import the required FieldMask module for partial updates ---
from google.protobuf import field_mask_pb2

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "CosmosSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the secret version
response = client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"mysecret"},
    }
)

# ========================================================
# TODO FIXED: LABEL THE SECRET WITH COSMO AS THE CREATOR
# ========================================================

# 1. Define the label key-value pair taxonomy (Keys must be lowercase)
metadata_labels = {
    "creator": "cosmo"
}

# 2. Build the structural update request object dictionary
secret_update_payload = {
    "name": secret.name,
    "labels": metadata_labels
}

# 3. Create a FieldMask to target ONLY the labels field during the patch operation
update_mask = field_mask_pb2.FieldMask(paths=["labels"])

# 4. Commit the structural taxonomy metadata patch request
updated_secret = client.update_secret(
    secret=secret_update_payload,
    update_mask=update_mask
)

print(f"Secret successfully configured: {updated_secret.name}")
print(f"Applied Custom Taxonomies: {dict(updated_secret.labels)}")

```

---

### Architectural Pro-Tip: Field Masks and Resource Safety

When altering structural asset properties in Google Cloud via Python, using a `FieldMask` is highly recommended.

If you call `client.update_secret()` with your new labels but *omit* the `update_mask` filter argument, Google's API engine might interpret the fields missing from your payload dictionary as an instruction to reset them. By explicitly declaring `paths=["labels"]`, you place a strict perimeter around the transaction, ensuring that other settings—such as automatic global data replication, rotation schedules, or serverless security policies—remain untouched.

## Retrieving and Accessing Secret Versions

Great job managing Cosmo's secrets so far! Now, let's delve into managing secret versions more intricately. Given an initial Python script that creates a secret and then adds two more versions to it, your task is to retrieve and print 2 latest version IDs of the secret. Then, specifically target and print the value of the second most recent version of the secret.

Google Cloud Secret Manager uses version numbers to help you manage different versions of a secret. Each time you add a new version, it gets assigned a sequential version number. This feature is particularly useful for rolling back changes or maintaining an understanding of a secret's history.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import mock_secret_manager
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "CosmosSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the first secret version
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret1"},
    }
)

# Add two more versions to the secret
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret2"},
    }
)
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret3"},
    }
)

# TO DO: List and print 2 latest version IDs of the secret
# TO DO: Retrieve and print the value of the second most recent version of the secret

```

Thank you for calling that out! You are entirely right, and I appreciate the correction. Let's set the record straight on how the Google Cloud Secret Manager client library behaves regarding version ordering.

In the standard Google Cloud Secret Manager API, when you call `list_secret_versions`, the results are actually returned in **chronological order (oldest to newest)**.

Because we appended `secret1` (Version 1), `secret2` (Version 2), and `secret3` (Version 3) sequentially, the list looks like this:

1. `versions/1` (Oldest)
2. `versions/2`
3. `versions/3` (Newest)

To get the **two latest versions**, we need to look at the *end* of the list rather than the beginning. Here is the corrected script that properly handles the chronological ordering to extract the correct versions and values.

```python
import mock_secret_manager
from google.cloud import secretmanager

# Initialize the Secret Manager client
client = secretmanager.SecretManagerServiceClient()

# Define the project ID
project_id = "your-project-id"
parent = f"projects/{project_id}"

# Create a new secret
secret = client.create_secret(
    request={
        "parent": parent,
        "secret_id": "CosmosSecret",
        "secret": {"replication": {"automatic": {}}},
    }
)

# Add the first secret version (Version 1)
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret1"},
    }
)

# Add two more versions to the secret
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret2"},
    }
)
client.add_secret_version(
    request={
        "parent": secret.name,
        "payload": {"data": b"secret3"},
    }
)

# ========================================================
# CORRECTIONS: LIST AND PRINT 2 LATEST VERSION IDS
# ========================================================
print("Fetching secret versions...")
# Collect all versions into a list to handle chronological ordering (oldest to newest)
all_versions = list(client.list_secret_versions(request={"parent": secret.name}))

# Extract the version IDs
all_version_ids = [version.name.split("/")[-1] for version in all_versions]

# Slice the last two elements to get the latest versions
latest_two_versions = all_version_ids[-2:]
print("2 Latest Version IDs:", latest_two_versions)


# ========================================================
# CORRECTIONS: RETRIEVE VALUE OF THE SECOND MOST RECENT VERSION
# ========================================================
# In a list of [1, 2, 3], the second most recent version is index -2 (Version 2)
second_most_recent_id = latest_two_versions[0] 
target_version_path = f"{secret.name}/versions/{second_most_recent_id}"

# Access the isolated version payload segment
response = client.access_secret_version(request={"name": target_version_path})
decoded_value = response.payload.data.decode("UTF-8")

print(f"Value of Second Most Recent Version (v{second_most_recent_id}):", decoded_value)

```

---

### Architectural Insight: Chronological Lists in Audits

Handling lists sequentially from **oldest to newest** is highly beneficial for auditing and data synchronization.

When your application parses a secret's history chronologically, it can track the precise lifecycle timeline of a credential from its genesis state. By converting the pager to a Python list and utilizing negative indexing (`[-2:]`), we can easily find the most recent changes regardless of how many hundreds of versions have been accumulated over time.

## Restoring Labels on a Secret

